In [16]:
import numpy as np
import paddle_quantum.qchem as qchem

In [17]:
def measure_times(gamma, delta, eps):
    return 2 * gamma**2 * np.log(2 / delta) / (eps**2)

def pro_cost_depo(p):
    return 1 / (1 - p)

def con_cost_depo(p):
    return (1 + p / 2) / (1 - p)

In [ ]:
# geo = qchem.geometry(structure=[['H', [-0., 0., 0.0]], ['H', [-0., 0., 0.74]]]) # H2
# geo = qchem.geometry(structure=[['H', [-0., 0., 0.0]], ['F', [-0., 0., 0.917]]]) # HF
geo = qchem.geometry(structure=[['O', [0., 0., -1.163]], ['C', [-0., 0, 0]], ['O', [-0., 0, 1.163]]]) # CO2

# generate molecular data
molecule = qchem.get_molecular_data(
    geometry=geo,
    basis='sto-3g',
    charge=0,
    multiplicity=1,
    method="fci",
    if_save=True,
    if_print=True
)
# construct the Hamiltonian
molecular_hamiltonian = qchem.spin_hamiltonian(molecule=molecule,
                                               filename=None, 
                                               multiplicity=1, 
                                               mapping_method='jordan_wigner')

In [19]:
weights = []
h_max = 0

for term, coefficient in zip(
    molecular_hamiltonian.pauli_words,
    molecular_hamiltonian.coefficients
):
    weight = sum(pauli in "XYZ" for pauli in term)
    # exclude the identity term
    if weight > 0:
        weights.append(weight)
        h_max = max(h_max, abs(coefficient))

weights = np.asarray(weights)

In [ ]:
# independent single-qubit depolarizing noise
p = 0.1

# accuracy requirements for each nonidentity term
delta = 0.01
eps = 0.01

gamma_pro = pro_cost_depo(p) ** weights
gamma_con = con_cost_depo(p) ** weights

# use h_max for every term
pro = np.sum(measure_times(gamma_pro * h_max, delta, eps))
con = np.sum(measure_times(gamma_con * h_max, delta, eps))

print("Proposed sampling total:", pro)
print("Conventional sampling total", con)